In [18]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import ttest_rel
import matplotlib.pyplot as plt

In [9]:
file = Path.cwd() / "results-survey998672.csv"

df = pd.read_csv(file, sep=";")

df.head()

,Antwort ID,Datum Abgeschickt,Letzte Seite,Start-Sprache,Zufallsstartwert,Datum gestartet,Datum letzte Aktivität,Bitte gib deine Initialen an. Diese Angabe dient ausschließlich der internen Zuordnung und erlaubt keine Identifikation deiner Person.,Was glaubst du wie nützlich ist ein KI-basierter Assistent mit Sprachausgabe in Rennspielen wie Forza Horizon 5? (0-10) [Nützlichkeit KI-basierter Assistent],"Das Experiment startet jetzt.Bitte fahre nun die vorgesehenen Rennen in Forza Horizon 5. Wenn du alle Rennen abgeschlossen hast und zur Umfrage zurückgekehrt bist, wähle unten „Ja“, um fortzufahren.",Wie gut passt die Post-Race Summary zu dem gefahrenen Rennen? (0-10) [Post-Race Summary],Ist das schlussendliche Feedback hilfreich? (0-10) [Feedbach hilfreich],Wie nützlich war der Assistent während des Rennens tatsächlich? (0-10) [Nützlichkeit Assistent während des Rennens],Wie nützlich ist die Ideallinie als Fahrhilfe gewesen? (0-10) [Ideallinie als Fahrhilfe]
0,1.0,19.02.2026 17:32,3.0,de,1.018061e+09,19.02.2026 16:47,19.02.2026 17:32,L.W.,8,Ja,7,8,4,7
1,2.0,19.02.2026 19:36,3.0,de,1.837495e+09,19.02.2026 18:33,19.02.2026 19:36,J.S.,8,Ja,6,4,6,9
2,3.0,19.02.2026 20:34,3.0,de,2.771647e+08,19.02.2026 20:22,19.02.2026 20:34,NB,5,Ja,8,8,7,9
3,4.0,20.02.2026 13:12,3.0,de,1.432777e+09,20.02.2026 12:18,20.02.2026 13:12,NS,7,Ja,6,8,9,7
4,5.0,21.02.2026 13:23,3.0,de,1.825153e+09,21.02.2026 11:30,21.02.2026 13:23,Am.L.,8,Ja,5,9,7,8


In [13]:
col_q0 = "Was glaubst du wie nützlich ist ein KI-basierter Assistent mit Sprachausgabe in Rennspielen wie Forza Horizon 5? (0-10) [Nützlichkeit KI-basierter Assistent]"

col_q1 = "Wie gut passt die Post-Race Summary zu dem gefahrenen Rennen? \xa0(0-10) [Post-Race Summary]"

col_q2 = "Ist das schlussendliche Feedback hilfreich? \xa0(0-10) [Feedbach hilfreich]"

col_q3 = "Wie nützlich war der Assistent während des Rennens tatsächlich? \xa0(0-10) [Nützlichkeit Assistent während des Rennens]"

col_q4 = "Wie nützlich ist die Ideallinie als Fahrhilfe gewesen? \xa0(0-10) [Ideallinie als Fahrhilfe]"

In [14]:
survey_cols = [col_q0, col_q1, col_q2, col_q3, col_q4]

for col in survey_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[survey_cols].isna().sum()

Was glaubst du wie nützlich ist ein KI-basierter Assistent mit Sprachausgabe in Rennspielen wie Forza Horizon 5? (0-10) [Nützlichkeit KI-basierter Assistent]    4
Wie gut passt die Post-Race Summary zu dem gefahrenen Rennen?  (0-10) [Post-Race Summary]                                                                        4
Ist das schlussendliche Feedback hilfreich?  (0-10) [Feedbach hilfreich]                                                                                         4
Wie nützlich war der Assistent während des Rennens tatsächlich?  (0-10) [Nützlichkeit Assistent während des Rennens]                                             4
Wie nützlich ist die Ideallinie als Fahrhilfe gewesen?  (0-10) [Ideallinie als Fahrhilfe]                                                                        4
dtype: int64

In [15]:
def descriptive_stats(series):
    mean = series.mean()
    std = series.std(ddof=1)   # sample SD
    sem = std / np.sqrt(len(series))
    ci_low = mean - 1.96 * sem
    ci_high = mean + 1.96 * sem
    
    return pd.Series({
        "Mean": mean,
        "Std_Dev": std,
        "SEM": sem,
        "CI_Lower": ci_low,
        "CI_Upper": ci_high
    })

results = pd.DataFrame()

for col in survey_cols:
    results[col] = descriptive_stats(df[col])

results = results.T
results.to_csv("csv/survey_descriptive_stats.csv")
results

,Mean,Std_Dev,SEM,CI_Lower,CI_Upper
Was glaubst du wie nützlich ist ein KI-basierter Assistent mit Sprachausgabe in Rennspielen wie Forza Horizon 5? (0-10) [Nützlichkeit KI-basierter Assistent],6.952381,1.716863,0.343373,6.279371,7.625391
Wie gut passt die Post-Race Summary zu dem gefahrenen Rennen? (0-10) [Post-Race Summary],7.238095,1.670472,0.334094,6.583270,7.892920
Ist das schlussendliche Feedback hilfreich? (0-10) [Feedbach hilfreich],7.285714,1.764734,0.352947,6.593939,7.977490
Wie nützlich war der Assistent während des Rennens tatsächlich? (0-10) [Nützlichkeit Assistent während des Rennens],5.523810,2.657424,0.531485,4.482099,6.565520
Wie nützlich ist die Ideallinie als Fahrhilfe gewesen? (0-10) [Ideallinie als Fahrhilfe],6.952381,2.036570,0.407314,6.154045,7.750717


In [16]:
t_stat, p_val = ttest_rel(df[col_q3], df[col_q4])

print("Assistent vs Ideallinie")
print("t =", t_stat)
print("p =", p_val)

Assistent vs Ideallinie
t = nan
p = nan


In [17]:
pairwise_tests = []

for i in range(len(survey_cols)):
    for j in range(i+1, len(survey_cols)):
        t_stat, p_val = ttest_rel(df[survey_cols[i]], df[survey_cols[j]])
        
        pairwise_tests.append({
            "Comparison": f"{survey_cols[i]} vs {survey_cols[j]}",
            "t_stat": t_stat,
            "p_value": p_val
        })

pairwise_results = pd.DataFrame(pairwise_tests)
pairwise_results.to_csv("survey_pairwise_tests.csv")
pairwise_results

,Comparison,t_stat,p_value
0,Was glaubst du wie nützlich ist ein KI-basiert...,NaN,NaN
1,Was glaubst du wie nützlich ist ein KI-basiert...,NaN,NaN
2,Was glaubst du wie nützlich ist ein KI-basiert...,NaN,NaN
3,Was glaubst du wie nützlich ist ein KI-basiert...,NaN,NaN
4,Wie gut passt die Post-Race Summary zu dem gef...,NaN,NaN
5,Wie gut passt die Post-Race Summary zu dem gef...,NaN,NaN
6,Wie gut passt die Post-Race Summary zu dem gef...,NaN,NaN
7,Ist das schlussendliche Feedback hilfreich? (...,NaN,NaN
8,Ist das schlussendliche Feedback hilfreich? (...,NaN,NaN
9,Wie nützlich war der Assistent während des Ren...,NaN,NaN


In [19]:
df = df.rename(columns={
    col_q0: "AI Usefulness (Pre)",
    col_q1: "Post-Race Fit",
    col_q2: "Final Feedback",
    col_q3: "Coach In-Race",
    col_q4: "Racing Line"
})

In [21]:
survey_plot_cols = [
    "AI Usefulness (Pre)",
    "Post-Race Fit",
    "Final Feedback",
    "Coach In-Race",
    "Racing Line"
]

plot_labels = ["AI (Pre)", "Summary", "Feedback", "Coach", "Line"]

data = [df[col].dropna() for col in survey_plot_cols]

plt.figure()

plt.boxplot(data)

plt.xticks(range(1, len(plot_labels) + 1), plot_labels)
plt.ylabel("Rating (0–10)")
plt.title("Survey Results")

plt.tight_layout()
plt.savefig("pictures/survey_boxplot.png")
plt.close()